# 00 Bronze - Explore the raw data

**Adventure Works Cycles** sells bikes, components, clothing and accessories through two channels: a web store
(individual customers) and resellers. This notebook series turns the raw data of the Fabric workshop into a
business-ready **star schema**, which Kenseme then turns into an ontology and a Data Agent.

## The medallion architecture

```
 BRONZE  (as landed)              SILVER  (clean, conformed)          GOLD  (business-ready)
 FabricWorkshopDemo               AdventureWorksSilver                AdventureWorksGold
 ───────────────────              ────────────────────                ──────────────────
 Files/data/sales.csv     ─┐      online_sales_line                   DimCustomer
 Files/bronze/2019-21.csv ─┼──►   reseller_order_line        ──►      DimProduct, DimProductCategory
 Tables/dbo/orders        ─┤      product                             DimSalesChannel, DimDate
 Files/products/...csv    ─┘      customer                            FactSalesOrder, FactSalesOrderLine
       00 explore                  01 Silver notebook                  02 Gold notebook   ──►  03 questions
```

| Layer | Purpose | Rule of thumb |
|---|---|---|
| **Bronze** | Keep the data exactly as it arrived, so any later step can be replayed | never edited, only read |
| **Silver** | One clean, typed, deduplicated table per business thing, with the same rules for every source | fix data quality here, once |
| **Gold** | Shaped for questions: facts (what happened) and dimensions (who, what, when, how) with keys | what BI tools, Kenseme and agents read |

**This notebook only reads.** It looks at the Bronze data and lists the problems Silver must fix.
Bronze is the workshop's own lakehouse **FabricWorkshopDemo**: it is not changed by any notebook of this series.

Run it cell by cell to tell the story, or use **Run all**.


In [ ]:
# Configuration. The helper finds a lakehouse in this workspace by its name.
from pyspark.sql import functions as F

def lakehouse_path(name):
    """OneLake path of a lakehouse in this workspace, found by name."""
    return notebookutils.lakehouse.get(name)["properties"]["abfsPath"]

BRONZE = lakehouse_path("FabricWorkshopDemo")  # read only

# The yearly files have no header row: these are the columns of sales.csv, in the same order.
SALES_COLS = ["SalesOrderNumber", "SalesOrderLineNumber", "OrderDate", "CustomerName", "EmailAddress",
              "Item", "Quantity", "UnitPrice", "TaxAmount"]

def read_csv(path, header=True):
    df = spark.read.option("header", header).option("quote", '"').option("escape", '"').csv(f"{BRONZE}/Files/{path}")
    return df if header else df.toDF(*SALES_COLS)

# Temporary views, so the SQL cells below can query the raw files by name
read_csv("data/sales.csv").createOrReplaceTempView("bronze_sales_csv")
(read_csv("bronze/2019.csv", False).withColumn("SourceFile", F.lit("2019.csv"))
 .unionByName(read_csv("bronze/2020.csv", False).withColumn("SourceFile", F.lit("2020.csv")))
 .unionByName(read_csv("bronze/2021.csv", False).withColumn("SourceFile", F.lit("2021.csv")))
 .createOrReplaceTempView("bronze_sales_yearly"))
read_csv("products/products.csv").createOrReplaceTempView("bronze_products_csv")
print("views: bronze_sales_csv, bronze_sales_yearly, bronze_products_csv; table: FabricWorkshopDemo.dbo.orders")

## 1. What landed in Bronze
Online sales arrived as CSV files, the product catalog as a CSV file, and reseller orders as a Delta table
(loaded by the workshop's dataflow).

In [ ]:
for folder in ["data", "bronze", "products"]:
    for f in notebookutils.fs.ls(f"{BRONZE}/Files/{folder}"):
        print(f"Files/{folder}/{f.name:<14} {f.size:>12,} bytes")

### Online sales lines (`sales.csv`)
One row per order line. There is **no customer id** (only name and e-mail) and **no product id** (only the
product name in `Item`). Every column is text until we type it.

In [ ]:
%%sql
SELECT * FROM bronze_sales_csv LIMIT 10

### The yearly files have no header
The first line of `2019.csv` is already data, so Spark would turn it into column names.

In [ ]:
display(spark.read.text(f"{BRONZE}/Files/bronze/2019.csv").limit(3))

## 2. Data quality findings
### Finding 1: every online line arrives twice
`sales.csv` holds all three years; the yearly files hold the same lines again.

In [ ]:
%%sql
SELECT 'sales.csv' AS source, COUNT(*) AS lines, COUNT(DISTINCT SalesOrderNumber) AS orders FROM bronze_sales_csv
UNION ALL
SELECT 'yearly files', COUNT(*), COUNT(DISTINCT SalesOrderNumber) FROM bronze_sales_yearly
UNION ALL
SELECT 'both, distinct order + line', COUNT(*), COUNT(DISTINCT SalesOrderNumber)
FROM (SELECT SalesOrderNumber, SalesOrderLineNumber FROM bronze_sales_csv
      UNION SELECT SalesOrderNumber, SalesOrderLineNumber FROM bronze_sales_yearly)

### Finding 2: customers have no id
The e-mail address is the only stable identifier of an online customer, so Silver uses it as the customer key.

In [ ]:
%%sql
SELECT COUNT(DISTINCT lower(trim(EmailAddress))) AS customers_by_email,
       COUNT(DISTINCT CustomerName)              AS distinct_names,
       MIN(to_date(OrderDate))                   AS first_order,
       MAX(to_date(OrderDate))                   AS last_order
FROM bronze_sales_csv

### Finding 3: products are referenced by name
Online lines name the product; the catalog has the id, category and list price. Every name must match one catalog row.

In [ ]:
%%sql
SELECT COUNT(DISTINCT s.Item)                                       AS products_sold,
       COUNT(DISTINCT CASE WHEN p.ProductName IS NULL THEN s.Item END) AS names_not_in_catalog
FROM bronze_sales_csv s LEFT JOIN bronze_products_csv p ON trim(p.ProductName) = trim(s.Item)

### Finding 4: one product name packs model, color and size
Questions like "top bike models" need the **model** on its own, so Silver splits the name.

In [ ]:
%%sql
SELECT ProductID, ProductName, Category, ListPrice FROM bronze_products_csv
WHERE ProductName LIKE 'Mountain-100%' OR ProductName LIKE 'HL Road Frame%' ORDER BY ProductName LIMIT 8

### Finding 5: reseller orders have a different shape
Numeric customer and product ids, a **line total** instead of a unit price, no tax, and an extra `MonthNo`
column. Resellers pay about 60% of list price, so the discount must be derived against the catalog.

In [ ]:
%%sql
SELECT * FROM FabricWorkshopDemo.dbo.orders ORDER BY SalesOrderID, LineItem LIMIT 10

In [ ]:
%%sql
SELECT COUNT(*) AS lines, COUNT(DISTINCT o.SalesOrderID) AS orders, MIN(o.OrderDate) AS first_order,
       MAX(o.OrderDate) AS last_order,
       ROUND(SUM(o.LineItemTotal) / SUM(o.OrderQty * CAST(p.ListPrice AS DOUBLE)), 3) AS paid_vs_list_price
FROM FabricWorkshopDemo.dbo.orders o JOIN bronze_products_csv p ON CAST(p.ProductID AS INT) = o.ProductID

## 3. What Silver will do

| Finding | Silver rule (notebook 01) |
|---|---|
| Online lines arrive twice | keep one row per order number + line number, prefer `sales.csv` |
| Everything is text | cast dates, integers and money (`decimal(19,4)`) |
| No customer id | online customer = lower-cased e-mail; reseller customer = `RS-<account id>` |
| Products referenced by name | keep the name to join the catalog, which has the id |
| Product name packs model, color, size | parse `ProductModel`, `Color`, `Variant`; add the category group |
| Reseller orders shaped differently | same column names as online, `SO` prefix on the order number, drop `MonthNo` |

Next: **01 Silver - Clean and conform**.